# Tiny Transformer Summarizer from Scratch
The objective of this notebook is to train a *Transformer* from scratch on a small dataset for text summarization. The model will be trained directly on the *text summarization* task. As in the last notebook, we will continue working on:
1. Data preparation
2. Building the LLM architecture
3. Training an LLM

However, there are two main differences with respect to the *transformer classifier* that we trained last time:
- *Encoder-Decoder architecture.* In order to be able to perform the text summarization task, we will train a transformer having an *encoder-decoder* type of architecture (or Seq2Seq architecture). The encoder part is quite similar to that of an encoder-only model, we will focus on the decoder part.
- *Maksing.* We will also focus on the masking technique. Masking is important to make sure that the decoder does not look into future tokens or the current token during training. We will also use masking in order to ignore the padding tokens that we will add to complete the different batches.

We start with the usual library imports.

> **Version complétée et expliquée.** Tous les `TODO` sont remplis. Les blocs « 💡 Explication » disent ce que fait chaque partie et pourquoi on a fait ces choix. Ce notebook réutilise beaucoup de briques du Notebook 3 : on ne réexplique que ce qui change.

### 💡 L'idée générale

Dans le Notebook 3, le modèle lisait une phrase et donnait **une étiquette** (une émotion). Ici, le modèle lit un **dialogue** et doit **écrire un texte** : son résumé. La sortie n'est plus une classe parmi 6, c'est une **suite de mots** de longueur variable.

On utilise l'architecture **encodeur-décodeur** du papier original des Transformers (celle des modèles de traduction, ou de BART et T5) :

```
dialogue ──► Encodeur ──► une représentation de chaque token du dialogue ─┐
                                                                          │ (cross-attention)
"<s> Bob asks"  ──► Décodeur ◄────────────────────────────────────────────┘ ──► prédit le mot suivant : "Alice"
```

- L'**encodeur** lit tout le dialogue d'un coup, comme au Notebook 3.
- Le **décodeur** écrit le résumé **mot par mot**. À chaque étape, il regarde :
  1. les mots du résumé déjà écrits (**self-attention masquée**) ;
  2. le dialogue, à travers la sortie de l'encodeur (**cross-attention**).
- Une couche linéaire finale donne un score pour **chacun des ~50 000 mots du vocabulaire**. Le mot avec le plus grand score est le mot suivant.

Les deux nouveautés du TP sont donc la **cross-attention** et le **masquage** (cacher le futur, et cacher le padding).

In [ ]:
import math

import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

💡 Comme au Notebook 3, il faut envoyer le modèle **et** les données sur `device`, sinon tout tourne sur CPU. C'est corrigé plus bas (création du modèle, `train_epoch`, `evaluate`).

## Encoder-decoder architecture
Since we are training a *transformer model* for the task of text summarization, an *encoder-decoder* transformer is necessary. Our encoder-decoder model will follow the classical transformer architecture, with the only difference that we will only be stacking a single encoder block and a single decoder block.

We will start by implementing all the necessary sub-modules.

### Multi-Head Attention
The *multi-head* attention layer takes the input and processes it in chunks of equal length through each of its different heads, by using the attention mechanism, i.e. computing the queries, keys and values, and the attention scores and weights from them.

We will modify the `MultiHeadAttention` class that we coded in the last notebook in order to:
- Allow for queries, keys and values coming from different input (important for the *decoder cross-attention* mechanism).
- Allow for *masking:* the `forward` method of the class will take an additional optional `mask` argument that we will apply to the attention scores before the softmax computation.

**Question.** Since we are masking attention scores *before* taking the softmax, what value should we replace the masked entries with?

**Exercise.** Choose an appropriate value for the masking process in the `attention` method of the class.

💡 **Explication : ce qui change dans l'attention.**

**1. Trois entrées `q`, `k`, `v` au lieu d'une seule `x`.**
- En **self-attention**, une séquence se regarde elle-même : on appelle `attention(x, x, x, mask)`.
- En **cross-attention** (dans le décodeur), les **queries viennent du résumé** en cours d'écriture, et les **keys et values viennent du dialogue** (la sortie de l'encodeur). Chaque mot du résumé va donc « chercher » l'information utile dans le dialogue. Les deux séquences n'ont pas la même longueur (64 tokens pour le résumé, 1024 pour le dialogue) : c'est pour ça que `forward` utilise `view(batch_size, -1, ...)`. Le `-1` laisse PyTorch déduire la longueur de chaque séquence.

**2. Le masque.** Un masque est un tableau de 1 (« tu as le droit de regarder ce token ») et de 0 (« interdit »). On s'en sert pour deux choses :
- **cacher le padding** : les `<pad>` ajoutés pour que toutes les séquences aient la même longueur ne contiennent aucune information ;
- **cacher le futur** dans le décodeur (voir le bloc décodeur plus bas).

**Réponse à la question : quelle valeur mettre aux positions masquées ?** On met **$-\infty$** (`float("-inf")`), parce que le masque est appliqué **avant le softmax** :

$$\text{softmax}(s)_j = \frac{e^{s_j}}{\sum_k e^{s_k}} \quad\text{et}\quad e^{-\infty} = 0$$

Une position à $-\infty$ reçoit donc un poids **exactement nul**, et les autres poids continuent de sommer à 1.

Pourquoi pas 0 ? Parce que $e^0 = 1$ : la position masquée garderait un poids normal, et le masque ne servirait à rien.

Attention à un piège : si **toute une ligne** était masquée, on aurait $\frac{0}{0}$ et donc des `NaN`. Ça n'arrive pas ici, parce que le premier token (`<s>`) n'est jamais masqué. Certaines implémentations utilisent une très grande valeur négative (`-1e9`) pour être à l'abri de ce cas.

`masked_fill_(mask == 0, valeur)` remplace **sur place** (le `_` final) les scores là où le masque vaut 0.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_dim, num_heads):
        super().__init__()
        assert hidden_dim % num_heads == 0, "hidden_dim must be divisible by num_heads"

        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads  # Compute the per-head hidden dimension

        self.W_query = nn.Linear(hidden_dim, hidden_dim) # queries weight matrix
        self.W_key = nn.Linear(hidden_dim, hidden_dim) # keys weight matrix
        self.W_value = nn.Linear(hidden_dim, hidden_dim) # values weight matrix

        self.W_out = nn.Linear(hidden_dim, hidden_dim)  # output weight matrix


    @staticmethod
    def attention(queries, keys, values, mask):
        head_dim = queries.shape[-1]

        # Compute attention scores
        attn_scores = torch.einsum("bijk, bikl -> bijl", queries, keys.transpose(2, 3))
        attn_scores = attn_scores / math.sqrt(head_dim)

        # Positions interdites (mask == 0) -> -inf : après le softmax leur poids vaut exactement 0
        if mask is not None:
            attn_scores.masked_fill_(mask == 0, float("-inf"))

        # Compute attention weights
        attn_weights = torch.softmax(attn_scores, dim=-1)

        # Attention output
        attn_output = torch.einsum("bijk, bikl -> bijl", attn_weights, values)

        # Return attention outputs (and attention weights for visualization purposes)
        return attn_output


    def forward(self, q, k, v, mask):
        batch_size, _, hidden_dim = q.shape
        assert hidden_dim == self.hidden_dim, f"hidden_dim must be {self.hidden_dim}"

        queries = self.W_query(q) # Shape: (batch_size, num_tokens, hidden_dim)
        keys = self.W_key(k)  # Shape: (batch_size, num_tokens, hidden_dim)
        values = self.W_value(v) # Shape: (batch_size, num_tokens, hidden_dim)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (batch_size, num_tokens, d_out) -> (batch_size, num_tokens, num_heads, head_dim)
        queries = queries.view(batch_size, -1, self.num_heads, self.head_dim)
        keys = keys.view(batch_size, -1, self.num_heads, self.head_dim)
        values = values.view(batch_size, -1, self.num_heads, self.head_dim)

        # Transpose: (batch_size, num_tokens, num_heads, head_dim) -> (batch_size, num_heads, num_tokens, head_dim)
        queries = queries.transpose(1, 2)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention)
        attn_output = self.attention(queries, keys, values, mask)

        # Transpose back: (batch_size, num_heads, num_tokens, head_dim) -> (batch_size, num_tokens, num_heads, head_dim)
        attn_output = attn_output.transpose(1, 2)

        # Concatenate heads: (batch_size, num_tokens, num_heads, head_dim) -> (batch_size, num_tokens, hiddend_dim)
        attn_output = attn_output.reshape(batch_size, -1, self.hidden_dim)

        # Compute output
        output = self.W_out(attn_output)

        return output

✅ **Petit test du masque.** On masque le dernier token d'une séquence de 4. Son poids d'attention doit être exactement 0, et les 3 autres poids doivent sommer à 1.

In [ ]:
q = torch.randn(1, 1, 4, 8)  # (batch, têtes, tokens, head_dim)
mask_test = torch.tensor([1, 1, 1, 0]).view(1, 1, 1, 4)  # le 4e token est du padding
scores = torch.einsum("bijk, bikl -> bijl", q, q.transpose(2, 3)) / math.sqrt(8)
scores.masked_fill_(mask_test == 0, float("-inf"))
print(torch.softmax(scores, dim=-1)[0, 0])  # dernière colonne = 0, chaque ligne somme à 1

In [ ]:
# %load solutions_nb4/masked_attention.py

### Layer Norm
The `LayerNorm` module is the same as lask time.

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.eps = 1e-5 # small value to avoid division by zero
        self.scale = nn.Parameter(torch.ones(hidden_dim)) # scale parameter (learnable)
        self.shift = nn.Parameter(torch.zeros(hidden_dim)) # shift parameter (learnable)

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        y = self.scale * norm_x + self.shift
        return y

### Feed-Forward Network
The *feed forward* module is the same as last time.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.ReLU(),
            nn.Linear(hidden_dim * 4, hidden_dim)
        )

    def forward(self, x):
        return self.layers(x)

### Transformer Encoder Block
We will keep the same architecture for the `TransformerEncoderBlock`as we did last time. However, we have changed the `MultiHeadAttention` class which now takes its inputs differently.

**Exercise.** Implement the `TransformerEncoderBlock` module below by completing the `TODO` tags.

💡 **Explication.** C'est le même bloc qu'au Notebook 3. La seule différence : `MultiHeadAttention` prend maintenant `(q, k, v, mask)`. Dans l'encodeur, c'est de la **self-attention** (le dialogue se regarde lui-même), donc on passe `x` trois fois. Le `mask` est le masque de padding du dialogue : aucun token ne prête attention aux `<pad>`.

Ici, le masque est indispensable. Les dialogues sont complétés jusqu'à **1024 tokens**, alors qu'un dialogue typique en fait 150 à 250. Sans masque, l'attention serait noyée dans des centaines de `<pad>`. C'était la limite qu'on avait signalée au Notebook 3.

In [ ]:
class TransformerEncoderBlock(nn.Module):
  def __init__(self, hidden_dim, num_heads):
    super().__init__()

    self.attention = MultiHeadAttention(hidden_dim, num_heads)
    self.norm1 = nn.LayerNorm(hidden_dim)
    self.norm2 = nn.LayerNorm(hidden_dim)
    self.feed_forward = FeedForward(hidden_dim)

  def forward(self, x, mask):
      attn_output = self.attention(x, x, x, mask)  # self-attention : queries = keys = values = x
      x = x + attn_output
      x = self.norm1(x)
      ff_output = self.feed_forward(x)
      x = x + ff_output
      x = self.norm2(x)
      return x

In [ ]:
# %load solutions_nb4/encoderblock.py

### Transformer Decoder Block
We will now build the *Transformer decoder block*, which is composed of:
1. A *Multi-head attention* sub-module, which will perform the self-attention on the decoder input, and will apply both a padding mask and a causal mask.
2. A *LayerNorm* normalization.
3. A *Multi-head attention* sub-module, which will perform the cross-attention between the decoder hidden state and the encoder hidden state, and will apply the encoder padding mask.
4. A *LayerNorm* normalization
5. A *Feed Forward* sub-module.
6. A *LayerNorm* normalization

Moreover, three skip-connections are present in the *Transformer Block*:
- *(A)* A skip connection that adds the original input to the output of the operation *1* above.
- *(B)* A skip connection that adds the output of the operation *2* to the output of the operation *3* above.
- *(C)* A skip connection that adds the output of the operation *4* to the output of the operation *5* above

**Exercise.** Implement the `TransformerDecoderBlock` module below by completing the `TODO` tags.

💡 **Explication : le bloc décodeur.**

```
résumé ─► Self-attention masquée ─► (+) ─► Norm ─► Cross-attention ─► (+) ─► Norm ─► FeedForward ─► (+) ─► Norm
  └──────────────────────────────────┘     │       (k, v = encodeur)  ▲      │                     ▲
              (A)                          └──────────────────────────┘      └─────────────────────┘
                                                      (B)                              (C)
```

C'est le bloc encodeur avec **une couche d'attention en plus** au milieu : la cross-attention.

**Le masque causal (« ne pas regarder le futur »).** Pendant l'entraînement, on donne au décodeur **tout le résumé d'un coup** et on lui demande de prédire, à chaque position, le mot **suivant**. Si la position 3 pouvait voir la position 4, elle n'aurait qu'à recopier la réponse : le modèle « tricherait » et n'apprendrait rien d'utile. Au moment de générer un vrai résumé, ces mots futurs n'existeront pas encore.

Le masque causal est une matrice **triangulaire** : la ligne $i$ (le token qui regarde) n'a le droit de voir que les colonnes $j \le i$. Pour 4 tokens :

```
         voit : t0 t1 t2 t3
token t0 :      1  0  0  0
token t1 :      1  1  0  0
token t2 :      1  1  1  0
token t3 :      1  1  1  1
```

Dans `add_causal_mask` :
- `torch.triu(ones, diagonal=1)` met des 1 **strictement au-dessus** de la diagonale (le futur) ;
- `== 0` inverse : 1 = autorisé (passé et présent), 0 = futur interdit ;
- `mask & causal_mask` combine avec le masque de padding. Une position est autorisée seulement si elle n'est **ni du futur, ni du padding**. Par broadcasting, `(b, 1, 1, s)` & `(1, 1, s, s)` donne `(b, 1, s, s)`.

**Correction d'un bug dans le code fourni :** la ligne `causal_mask.to(device)` ne fait rien, car `.to()` **renvoie** un nouveau tenseur sans modifier l'ancien. Il faut écrire `causal_mask = causal_mask.to(...)`. Sur GPU, l'oubli provoque l'erreur *« Expected all tensors to be on the same device »*. On utilise `mask.device` plutôt que la variable globale `device`, pour que le masque soit toujours là où sont les données.

**Quels masques pour quelle attention ?**
- self-attention : `tgt_mask` (padding du résumé **+** causal) ;
- cross-attention : `src_mask` (padding du **dialogue**), car ce sont les keys du dialogue qu'on doit filtrer. Pas besoin de masque causal ici : le dialogue est entièrement connu, il n'y a pas de « futur » à cacher.

In [ ]:
class TransformerDecoderBlock(nn.Module):
    def __init__(self, hidden_dim, num_heads):
        super().__init__()

        self.self_attention = MultiHeadAttention(hidden_dim, num_heads)  # le résumé se regarde lui-même
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.cross_attention = MultiHeadAttention(hidden_dim, num_heads)  # le résumé regarde le dialogue
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.feed_forward = FeedForward(hidden_dim)
        self.norm3 = nn.LayerNorm(hidden_dim)

    @staticmethod
    def add_causal_mask(mask):
        # assuming attn scores of shape (batch_size, num_heads, seq_len, seq_len)
        # assuming mask of shape (batch_size, 1, 1, seq_len)
        causal_mask = torch.triu(torch.ones(1, 1, mask.size(-1), mask.size(-1)), diagonal=1).type(torch.int)  # shape (1, 1, seq_len, seq_len)
        causal_mask = causal_mask == 0
        causal_mask = causal_mask.to(mask.device)  # .to() renvoie un nouveau tenseur : il faut le réassigner
        return mask & causal_mask

    def forward(self, x, enc_hidden_state, src_mask, tgt_mask):
        tgt_mask = self.add_causal_mask(tgt_mask)  # padding du résumé + interdiction de voir le futur

        # (A) self-attention masquée sur le résumé
        x = self.norm1(x + self.self_attention(x, x, x, tgt_mask))
        # (B) cross-attention : queries = résumé, keys/values = sortie de l'encodeur (le dialogue)
        x = self.norm2(x + self.cross_attention(x, enc_hidden_state, enc_hidden_state, src_mask))
        # (C) feed-forward
        x = self.norm3(x + self.feed_forward(x))
        return x

✅ **Visualisation du masque combiné** pour un résumé de 5 tokens dont les 2 derniers sont du padding :

In [ ]:
padding_mask = torch.tensor([[1, 1, 1, 0, 0]]).view(1, 1, 1, 5)
print(TransformerDecoderBlock.add_causal_mask(padding_mask)[0, 0].int())
# Ligne i = ce que le token i a le droit de regarder : seulement le passé, et jamais les 2 tokens de padding

In [ ]:
# %load solutions_nb4/decoderblock.py

### Embedding
The `Embedding` module is the same as for the encoder-only model of last time.

In [ ]:
class Embedding(nn.Module):
  def __init__(self, vocab_size, max_length, hidden_dim):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, hidden_dim)
    self.position_encoding = nn.Embedding(max_length, hidden_dim)

  def forward(self, x):
    _, seq_length = x.shape
    token_embeddings = self.embedding(x)
    pos_encodings = self.position_encoding(torch.arange(seq_length, device=x.device))
    return token_embeddings + pos_encodings

### Transformer Encoder-Decoder Model for Text Summarization
We have implemented all necessary sub-modules ane we are now ready to implement the *end-to-end* architecture of the *transformer encoder-decoder* model. To that end, we will make use of:
- The `Embedding` sub-modules (one for the encoder, one for the decoder).
- The `TransformerEncoderBlock` sub-module
- The `TransformerDecoderBlock` sub-module
- A `nn.Linear` layer to act as a classification head.

**Note.** The output of the `TransformerDecoderBlock` sub-module is a tensor of shape $(b, s, d_h)$ where $b$ represents the batch size, $s$ the sequence length, and $d_h$ the hidden dimension of the model. In order to perform the classification task, we need to compute one logit per token and per sequence in the batch.

**Exercise.** Implement the `TransformerSummarizer` model by completing the `TODO` tags below.

💡 **Explication : le modèle complet.**

- **Deux embeddings** : un pour le dialogue (`src`, positions jusqu'à 1024) et un pour le résumé (`tgt`, positions jusqu'à 64). Ils partagent le même vocabulaire (même tokenizer), mais pas le même nombre de positions. On en utilise deux séparés, comme dans l'énoncé.
- **`encode`** : embedding du dialogue, puis bloc encodeur → `(b, 1024, 256)`.
- **`decode`** : embedding du résumé, puis bloc décodeur, qui reçoit aussi la sortie de l'encodeur → `(b, 64, 256)`.
- **`summarizer_head`** : `nn.Linear(256, vocab_size)`. Contrairement au Notebook 3, on ne prend pas seulement `[CLS]` : on calcule des logits **pour chaque position** du résumé, car à chaque position il faut prédire **le mot suivant parmi tout le vocabulaire**. C'est une classification à ~50 000 classes, répétée 64 fois. La sortie est `(b, 64, vocab_size)`.

**Les masques et `unsqueeze`.** Le tokenizer donne des masques de forme `(b, s)`. Les scores d'attention ont la forme `(b, têtes, s_query, s_key)`. `unsqueeze(1).unsqueeze(2)` transforme le masque en `(b, 1, 1, s_key)` : le broadcasting applique alors le même masque **à toutes les têtes et à toutes les queries**, et seules les **keys** (les tokens qu'on regarde) sont masquées.

**Pourquoi `encode` et `decode` séparés ?** Pour la génération (à la fin du notebook) : on encode le dialogue **une seule fois**, puis on appelle `decode` à chaque nouveau mot.

In [ ]:
class TransformerSummarizer(nn.Module):
    def __init__(self,
               vocab_size,
               encoder_max_length,
               decoder_max_length,
               hidden_dim,
               num_heads):
        super().__init__()

        self.src_embedding = Embedding(vocab_size, encoder_max_length, hidden_dim)  # dialogue : jusqu'à 1024 positions
        self.tgt_embedding = Embedding(vocab_size, decoder_max_length, hidden_dim)  # résumé : jusqu'à 64 positions
        self.encoder = TransformerEncoderBlock(hidden_dim, num_heads)
        self.decoder = TransformerDecoderBlock(hidden_dim, num_heads)
        self.summarizer_head = nn.Linear(hidden_dim, vocab_size)  # un score par mot du vocabulaire


    def encode(self, x_src, src_mask):
        src_mask = src_mask.unsqueeze(1).unsqueeze(2) # (b, seq_len) -> (b, 1, 1, seq_len)
        x_src = self.src_embedding(x_src)  # (b, src_len) -> (b, src_len, hidden_dim)
        x_src = self.encoder(x_src, src_mask)  # (b, src_len, hidden_dim)
        return x_src


    def decode(self, x_tgt, encoder_hidden_state, src_mask, tgt_mask):
        src_mask = src_mask.unsqueeze(1).unsqueeze(2) # (b, seq_len) -> (b, 1, 1, seq_len)
        tgt_mask = tgt_mask.unsqueeze(1).unsqueeze(2) # (b, seq_len) -> (b, 1, 1, seq_len)
        x_tgt = self.tgt_embedding(x_tgt)  # (b, tgt_len) -> (b, tgt_len, hidden_dim)
        x_tgt = self.decoder(x_tgt, encoder_hidden_state, src_mask, tgt_mask)  # (b, tgt_len, hidden_dim)
        return x_tgt


    def forward(self, x_src, x_tgt, src_mask, tgt_mask):
        encoder_hidden_state = self.encode(x_src, src_mask)
        decoder_hidden_state = self.decode(x_tgt, encoder_hidden_state, src_mask, tgt_mask)
        logits = self.summarizer_head(decoder_hidden_state)  # (b, tgt_len, vocab_size)
        return logits

In [ ]:
# %load solutions_nb4/encoderdecoder.py

## Data Preparation
We will train the `TransformerSummarizer` model on a text summarization task. The data set we will be using is the following [dataset](https://huggingface.co/datasets/knkarthick/dialogsum). It consists of pairs of text of type `document` and `summary`.

In [ ]:
from datasets import load_dataset, DatasetDict

raw_dataset = load_dataset("knkarthick/dialogsum")
raw_dataset

**Questions.** Describe the datasets above.

**Exercise.** Print one of the elements of ``raw_dataset["train"]``, both the document and its summary.

💡 **Réponse : description des données.** *DialogSum* contient des **dialogues de la vie courante** (rendez-vous chez le médecin, entretien d'embauche, achats…) entre deux personnes ou plus, anonymisées en `#Person1#`, `#Person2#`… Chaque dialogue est accompagné d'un **résumé écrit à la main**, en 1 à 3 phrases, à la troisième personne.

`raw_dataset` est un `DatasetDict` avec 3 découpages (les tailles sont affichées au-dessus) : `train` (~12 500 dialogues), `validation` (500) et `test` (1 500). Chaque exemple a 4 colonnes :
- `id` : un identifiant ;
- `dialogue` : le texte source, l'**entrée de l'encodeur** ;
- `summary` : le résumé de référence, ce que le **décodeur doit apprendre à écrire** ;
- `topic` : le thème du dialogue (non utilisé ici).

Le code ci-dessous affiche un exemple et des statistiques de longueur. Elles justifient les tailles maximales choisies plus bas (1024 tokens pour le dialogue, 64 pour le résumé).

In [ ]:
example = raw_dataset["train"][0]
print("Document:")
print(example["dialogue"])
print("")
print("Summary:")
print(example["summary"])
print("")
print("Topic:", example["topic"])

In [ ]:
import numpy as np

# Longueur des dialogues et des résumés, en nombre de mots, sur le train
dialogue_lengths = np.array([len(d.split()) for d in raw_dataset["train"]["dialogue"]])
summary_lengths = np.array([len(s.split()) for s in raw_dataset["train"]["summary"]])
print(f"Dialogue : {dialogue_lengths.mean():.0f} mots en moyenne, max {dialogue_lengths.max()}")
print(f"Résumé   : {summary_lengths.mean():.0f} mots en moyenne, max {summary_lengths.max()}")
print(f"Un résumé fait en moyenne {summary_lengths.mean() / dialogue_lengths.mean():.0%} de la longueur du dialogue")

In [ ]:
# %load solutions_nb4/print_elements.py

The above `raw_dataset` is quite large, and training the transformer on it will take too much time. In order to be able to carry our experiments in a reasonable time, we will reduce the size of the dataset.

**Exercise.** Create a `tiny_dataset` by keeping the first 500 training exmamples, the first 100 validation examples, and the first 100 test examples.

💡 **Explication.** Entraîner sur les 12 500 dialogues prendrait trop de temps pour un TP. On garde les **premiers** exemples de chaque découpage avec `select(range(n))`, puis on reconstruit un `DatasetDict` avec la même structure que l'original. Tout le code qui suit fonctionne donc de la même façon.

⚠️ Conséquence à garder en tête : **500 exemples, c'est très peu** pour apprendre à écrire de l'anglais à partir de zéro. Il ne faut pas s'attendre à des résumés de qualité ; le but est de comprendre le mécanisme.

In [ ]:
tiny_dataset = DatasetDict({
    "train": raw_dataset["train"].select(range(500)),  # les 500 premiers exemples d'entraînement
    "validation": raw_dataset["validation"].select(range(100)),
    "test": raw_dataset["test"].select(range(100)),
})
print(tiny_dataset)

In [ ]:
# %load solutions_nb4/tinydataset.py

### Pre-trained Tokenizer
In order to convert the textual data in the above datasets into a format that can be used as input for our models, we first *tokenize* the text using a pre-trained tokenizer.

In [ ]:
from transformers import BartTokenizer

tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")

💡 **Explication.** On prend le tokenizer de **BART**, un modèle encodeur-décodeur conçu justement pour le résumé. C'est un tokenizer **BPE au niveau des octets** (comme GPT-2) : il garde les majuscules et les espaces (un espace devient `Ġ` collé au mot suivant), et il peut représenter n'importe quel texte sans token inconnu.

Comme au Notebook 3, on ne prend **que le tokenizer**. Notre modèle part de zéro.

**Exercise.** Let's find out more about the tokenizer we will be using. Write code in order to answer the following questions:
1. What is the name of the tokenizer being used?
2. What is the size of the vocabulary?
3. What is the maximum model input length?
4. What special tokens does the tokenizer use? What are their IDs?

In [ ]:
print("1. Nom du tokenizer :", tokenizer.name_or_path, "|", type(tokenizer).__name__)
print("2. Taille du vocabulaire :", tokenizer.vocab_size)
print("3. Longueur max d'une entrée :", tokenizer.model_max_length)
print("4. Tokens spéciaux et leurs IDs :")
for name, token in tokenizer.special_tokens_map.items():
    print(f"   {name:10s} {str(token):8s} -> {tokenizer.convert_tokens_to_ids(token)}")

encoded = tokenizer("#Person1#: Hi, Mr. Smith.")
print()
print("IDs    :", encoded["input_ids"])
print("Tokens :", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))

💡 **Ce qu'il faut retenir :**
- vocabulaire de **50 265** tokens : c'est la taille de la sortie du modèle, puisqu'il choisit le mot suivant parmi tous ces tokens ;
- longueur max : selon la version de `transformers`, on voit soit **1024**, soit un nombre énorme (~10³⁰). Le nombre énorme est une valeur par défaut qui veut dire « non renseigné dans la config du tokenizer ». La vraie limite de BART est **1024 positions**, d'où `ENCODER_MAX_LEN = 1024` ;
- le nom de classe affiché peut être `RobertaTokenizer` : BART utilise exactement le même tokenizer que RoBERTa ;
- tokens spéciaux :
  - `<s>` = 0 : **début** de séquence (*BOS*), joue le rôle du `[CLS]` de BERT ;
  - `<pad>` = 1 : padding ;
  - `</s>` = 2 : **fin** de séquence (*EOS*), c'est en le produisant que le modèle signale que son résumé est terminé ;
  - `<unk>` = 3 : token inconnu ;
  - `<mask>` = 50264 : utilisé pour pré-entraîner BART, pas ici.
- On voit dans l'exemple que `#Person1#` est découpé en plusieurs morceaux, et que le tokenizer ajoute automatiquement `<s>` au début et `</s>` à la fin.

In [ ]:
# %load solutions_nb4/tokenizer.py

### Pre-processing the data
We next build a pre-processing function in order to:
1. Tokenize the document and summaries.
2. Generate the appropriate inputs and outputs.

Note that our model will be trained using:
- The document as the input to the encoder.
- The summary as the input to the decoder.
- The summary with the tokens shifter right as the target output.

In order to produce the decoder input-output pairs, we will suppress the *end-of-sequence* token from the decoder input, and the *beginning-of-sequence* token from the target output.

💡 **Explication : le « teacher forcing » et le décalage d'un cran.**

Le décodeur apprend à **prédire le token suivant**. Pour un résumé tokenisé `<s> Bob calls Alice </s>`, on construit deux séquences **décalées d'une position** :

| position | 0 | 1 | 2 | 3 |
|---|---|---|---|---|
| **entrée du décodeur** (`decoder_input_ids`) = résumé **sans `</s>`** | `<s>` | Bob | calls | Alice |
| **cible** (`labels`) = résumé **sans `<s>`** | Bob | calls | Alice | `</s>` |

À la position $i$, le décodeur voit les tokens $0..i$ de l'entrée (grâce au masque causal) et doit prédire `labels[i]`, c'est-à-dire le token $i+1$ du résumé. Le dernier mot à prédire est `</s>` : le modèle apprend ainsi **quand s'arrêter**.

On appelle ça le **teacher forcing** : pendant l'entraînement, on donne au décodeur les **vrais** mots précédents, pas ceux qu'il a lui-même prédits. Les 64 prédictions d'un résumé se font donc **en parallèle**, en un seul passage. C'est ce qui rend l'entraînement des Transformers rapide.

**Dans le code :**
- le dialogue est complété ou tronqué à exactement 1024 tokens (`padding='max_length'`), avec son masque de padding ;
- le résumé est tokenisé sur **65** tokens (`DECODER_MAX_LEN + 1`). En retirant un token de chaque côté, on obtient bien 64 tokens pour l'entrée comme pour la cible ;
- `eos_index` repère où est le `</s>` pour le retirer de l'entrée du décodeur (s'il y a du padding, il est **avant** les `<pad>`, pas à la fin).

In [ ]:
VOCAB_SIZE = tokenizer.vocab_size
ENCODER_MAX_LEN = 1024
DECODER_MAX_LEN = 64

In [ ]:
def preprocessing_function(example):
    model_inputs = {}
    encoder_inputs = tokenizer(example["dialogue"], max_length=ENCODER_MAX_LEN, padding='max_length', truncation=True)
    model_inputs["encoder_input_ids"] = encoder_inputs["input_ids"]
    model_inputs["encoder_mask"] = encoder_inputs["attention_mask"]
    labels = tokenizer(example["summary"], max_length=DECODER_MAX_LEN + 1, padding='max_length', truncation=True)
    eos_index = labels["input_ids"].index(tokenizer.eos_token_id)
    model_inputs["decoder_input_ids"] = labels["input_ids"][:eos_index]+labels["input_ids"][eos_index+1:]
    model_inputs["decoder_mask"] = labels["attention_mask"][:eos_index] + labels["attention_mask"][eos_index+1:]
    model_inputs["labels"] = labels["input_ids"][1:]
    return model_inputs

In [ ]:
# Apply tokenization to the entire dataset
tokenized_dataset = tiny_dataset.map(preprocessing_function, batched=False)

✅ **Vérification du décalage** sur le premier exemple : la cible est bien l'entrée décalée d'un cran.

In [ ]:
ex = tokenized_dataset["train"][0]
n = 8
print("Entrée décodeur :", tokenizer.convert_ids_to_tokens(ex["decoder_input_ids"][:n]))
print("Cible (labels)  :", tokenizer.convert_ids_to_tokens(ex["labels"][:n]))
print("Longueurs :", len(ex["encoder_input_ids"]), len(ex["decoder_input_ids"]), len(ex["labels"]))
print("Tokens réels dans le dialogue :", sum(ex["encoder_mask"]), "sur", ENCODER_MAX_LEN)

In [ ]:
# Remove the original text fields to save memory
tokenized_dataset = tokenized_dataset.remove_columns(["dialogue", "summary", "id"])

# Print dataset structure
print(tokenized_dataset)

### Data Loaders
Finally, we create three separate pytorch data loaders in order to easily iterate through them during the training and evaluation phases.

In [ ]:
tokenized_dataset.set_format(type="torch")

train_loader = torch.utils.data.DataLoader(
    tokenized_dataset["train"],
    shuffle=True,
    batch_size=16,
)

val_loader = torch.utils.data.DataLoader(
    tokenized_dataset["validation"],
    shuffle=False,
    batch_size=16,
)

test_loader = torch.utils.data.DataLoader(
    tokenized_dataset["test"],
    shuffle=False,
    batch_size=16,
)

💡 Ici, pas besoin de `DataCollatorWithPadding` comme au Notebook 3 : toutes les séquences ont **déjà** la même longueur (1024 et 64), grâce à `padding='max_length'`. C'est plus simple, mais plus coûteux, car l'encodeur calcule l'attention sur 1024 × 1024 positions même pour un dialogue de 200 tokens. Le masque garantit que ces `<pad>` n'influencent pas le résultat. `batch_size=16` : 16 dialogues traités à la fois.

⚠️ **Si tu obtiens `CUDA out of memory`** (GPU avec peu de mémoire), passe `batch_size` à 8 ou 4 dans les trois DataLoaders, puis redémarre le noyau. Chaque attention de l'encodeur stocke une matrice 16 × 8 têtes × 1024 × 1024, soit 512 Mo, et plusieurs de ces matrices sont gardées en mémoire pour la rétropropagation.

## Training and Evaluation
Now that we have both our model architecture defined and our data prepared, we can proceed to the last phase of the lab project: training and evaluating the model.

### Initialize model

In [ ]:
HIDDEN_DIM = 256
NUM_HEADS = 8

print(f"VOCAB_SIZE: {VOCAB_SIZE}")
print(f"ENCODER_MAX_LEN: {ENCODER_MAX_LEN}")
print(f"DECODER_MAX_LEN: {DECODER_MAX_LEN}")
print(f"HIDDEN_DIM: {HIDDEN_DIM}")
print(f"NUM_HEADS: {NUM_HEADS}")

In [ ]:
torch.manual_seed(0)  # initialisation reproductible
summarizer = TransformerSummarizer(VOCAB_SIZE, ENCODER_MAX_LEN, DECODER_MAX_LEN, HIDDEN_DIM, NUM_HEADS).to(device)
print(summarizer)
print(f"\nNombre de paramètres : {sum(p.numel() for p in summarizer.parameters()):,}")

💡 **Lecture.** Environ **39 millions de paramètres**, dont la grande majorité vient du vocabulaire de 50 265 tokens :
- 2 embeddings de 50 265 × 256 ≈ 2 × 12,9 M ;
- la tête de sortie, 256 × 50 265 ≈ 12,9 M.

Les blocs encodeur et décodeur ne pèsent qu'environ 2 M. C'est beaucoup de paramètres à apprendre avec seulement 500 exemples, et c'est pour ça qu'il ne faudra pas attendre de miracles.

### Train Model
We can now proceed to the model training phase. In order to do so, we will define two functions:
- A `train_epoch` function that will train the model by iterating through the given data loader once.
- An `evaluate` function that will evaluate the model on the given data loader.

💡 **Ce qui change par rapport au Notebook 3 :**
- le modèle sort `(b, 64, vocab_size)` et les labels sont `(b, 64)`. `CrossEntropyLoss` attend `(N, classes)` et `(N,)`, donc on **aplatit** avec `view(-1, VOCAB_SIZE)` et `view(-1)` : chacune des b × 64 positions devient un exemple de classification ;
- on ajoute `.to(device)` à toutes les entrées du batch, comme au Notebook 3.

**Une correction importante : ignorer le padding dans la loss et l'accuracy.** La plupart des résumés font 20 à 30 tokens. Les ~35 positions restantes des labels sont des `<pad>`. Si on les compte :
- le modèle apprend surtout à prédire `<pad>`, ce qui est facile et inutile ;
- l'accuracy est **gonflée** : un modèle qui ne prédit que `<pad>` aurait déjà ~50 % !

On utilise donc `nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)`. Les positions dont le label vaut `<pad>` sont exclues du calcul de la loss. De la même façon, l'accuracy ne compte que les positions **non-padding**. On garde le `</s>` dans le calcul, car apprendre à s'arrêter fait partie de la tâche.

In [ ]:
import time
from tqdm import tqdm

In [ ]:
def train_epoch(model, dataloader, optimizer, criterion):
    epoch_loss = 0
    epoch_acc = 0

    model.train()

    for batch in tqdm(dataloader, desc="Processing Batches"):
        optimizer.zero_grad()

        encoder_input_ids = batch["encoder_input_ids"].to(device)
        encoder_mask = batch["encoder_mask"].to(device)
        decoder_input_ids = batch["decoder_input_ids"].to(device)
        decoder_mask = batch["decoder_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(encoder_input_ids, decoder_input_ids, encoder_mask, decoder_mask)
        outputs = outputs.view(-1, VOCAB_SIZE)  # (b * 64, vocab_size)
        labels = labels.view(-1)  # (b * 64,)
        loss = criterion(outputs, labels)

        not_pad = labels != tokenizer.pad_token_id  # on ne compte que les vrais tokens
        acc = (outputs.argmax(dim=1)[not_pad] == labels[not_pad]).float().mean()

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        epoch_acc += acc.item()

    return epoch_loss / len(dataloader), epoch_acc / len(dataloader)

In [ ]:
def evaluate(model, dataloader, criterion):
    epoch_loss = 0
    epoch_acc = 0
    model.eval()

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Processing Batches"):

            encoder_input_ids = batch["encoder_input_ids"].to(device)
            encoder_mask = batch["encoder_mask"].to(device)
            decoder_input_ids = batch["decoder_input_ids"].to(device)
            decoder_mask = batch["decoder_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(encoder_input_ids, decoder_input_ids, encoder_mask, decoder_mask)
            outputs = outputs.view(-1, VOCAB_SIZE)
            labels = labels.view(-1)
            loss = criterion(outputs, labels)

            not_pad = labels != tokenizer.pad_token_id
            acc = (outputs.argmax(dim=1)[not_pad] == labels[not_pad]).float().mean()

            epoch_loss += loss.item()
            epoch_acc += acc.item()

    return epoch_loss / len(dataloader), epoch_acc / len(dataloader)

💡 **Lire les chiffres.**
- Avec 50 265 mots possibles, une prédiction au hasard donne une loss de $\ln(50\,265) \approx 10{,}8$ : c'est le point de départ.
- L'**accuracy** ici, c'est le pourcentage de tokens du résumé où le mot suivant prédit est **exactement** le bon, en connaissant les vrais mots précédents (teacher forcing). C'est une mesure sévère : *"Bob phones Alice"* au lieu de *"Bob calls Alice"* compte comme une erreur.
- Les vrais résumés automatiques s'évaluent plutôt avec **ROUGE**, qui mesure les mots en commun avec le résumé de référence.

In [ ]:
EPOCHS = 5
LEARNING_RATE = 1e-3

optimizer = torch.optim.Adam(summarizer.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)  # le padding ne compte pas dans la loss

for epoch in range(EPOCHS):
    start_time = time.time()

    train_loss, train_acc = train_epoch(summarizer, train_loader, optimizer, criterion)
    valid_loss, valid_acc = evaluate(summarizer, val_loader, criterion)

    epoch_time = time.time() - start_time  # durée de l'epoch (l'énoncé affichait l'heure absolue)

    print("")
    print(f'Epoch: {epoch+1:02} | Time: {epoch_time:.0f}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. Acc: {valid_acc*100:.2f}%')

💡 **Ce qu'on observe.** Voici les chiffres de notre exécution de contrôle (faite avec `batch_size=4` faute de mémoire GPU ; avec 16, tes chiffres seront un peu différents, mais la tendance devrait être la même) :

| Epoch | Train loss | Val loss | Train acc | Val acc |
|---|---:|---:|---:|---:|
| 1 | 5,42 | 4,57 | 31 % | 37 % |
| 2 | 3,92 | **4,49** | 36 % | 38 % |
| 3 | 3,23 | 4,54 | 40 % | 37 % |
| 4 | 2,51 | 4,65 | 47 % | 39 % |
| 5 | 1,81 | 4,94 | 59 % | 37 % |

C'est un cas d'école de **sur-apprentissage** (*overfitting*) :
- la loss de **train** baisse sans arrêt (5,4 → 1,8) et l'accuracy de train monte (31 % → 59 %) : le modèle apprend de mieux en mieux **les 500 résumés d'entraînement** ;
- la loss de **validation** atteint son minimum dès l'**epoch 2**, puis **remonte** (4,49 → 4,94), et l'accuracy de validation stagne autour de 37 %. Sur des dialogues qu'il n'a jamais vus, le modèle ne progresse plus.

Le modèle a ~39 M de paramètres pour seulement 500 exemples : il a largement de quoi **apprendre les résumés par cœur** au lieu d'apprendre à résumer. Les 37 % d'accuracy en validation viennent surtout de tokens très fréquents et faciles à deviner (`#`, `Person`, `1`, `#`, `.`, *and*, *the*…).

**Ce qu'il faudrait faire :**
- s'arrêter à l'epoch où la loss de validation est minimale (*early stopping*), ici la 2ᵉ ;
- surtout, utiliser **beaucoup plus de données** : le dataset complet a 12 460 dialogues, soit 25 fois plus ;
- ajouter de la régularisation (dropout), ou partir d'un modèle **pré-entraîné** (BART), ce qui est la vraie solution en pratique.

### Evaluation
Finally, we evaluate the model on the test set.

In [ ]:
test_loss, test_acc = evaluate(summarizer, test_loader, criterion)
print("")
print(f'Test Loss: {test_loss:.3f} |  Test Acc: {test_acc*100:.2f}%')

💡 **Interprétation.** Le test confirme la validation. Dans notre exécution, la loss de test vaut ~5,1 et l'accuracy ~35 %, soit un peu moins bien qu'en validation à l'epoch 2, puisque le modèle final est celui de l'epoch 5, qui a sur-appris. On est loin du hasard (une loss de 10,8 correspondrait à des prédictions au hasard), donc le modèle a appris **quelque chose** : le style des résumés et les mots fréquents. Mais l'accuracy token par token ne dit pas si les résumés **ont du sens**. Pour ça, il faut en générer quelques-uns et les lire, ce que propose le dernier exercice.

**Exercise.** Try the model in a few test examples to check its performance.

💡 **Comment générer un résumé ? (décodage glouton, ou *greedy*)**

À l'entraînement, le décodeur recevait le vrai résumé. Pour un **nouveau** dialogue, on n'a pas de résumé : il faut l'écrire **un token à la fois**, en réinjectant chaque prédiction comme entrée. C'est la génération **autorégressive** :

```
étape 1 : entrée = [<s>]                  -> prédit "Bob"
étape 2 : entrée = [<s>, Bob]             -> prédit "calls"
étape 3 : entrée = [<s>, Bob, calls]      -> prédit "Alice"
...       jusqu'à ce que le modèle prédise </s>, ou qu'on atteigne 64 tokens
```

1. On encode le dialogue **une seule fois** avec `encode`.
2. À chaque étape, on décode la séquence déjà écrite. On ne garde que les logits de la **dernière position** (le mot suivant), et on prend le plus probable (`argmax`). C'est le décodage **glouton**.
3. Le masque du résumé est rempli de 1, puisqu'il n'y a pas de padding dans ce qu'on génère. Le masque causal est ajouté automatiquement dans le décodeur.

(Des méthodes plus élaborées existent : le *beam search* garde plusieurs hypothèses en parallèle, l'échantillonnage avec température tire le mot au hasard selon les probabilités.)

In [ ]:
@torch.no_grad()
def generate_summary(model, dialogue, max_new_tokens=DECODER_MAX_LEN):
    model.eval()
    src = tokenizer(dialogue, max_length=ENCODER_MAX_LEN, padding="max_length", truncation=True, return_tensors="pt")
    src_ids = src["input_ids"].to(device)
    src_mask = src["attention_mask"].to(device)
    encoder_hidden_state = model.encode(src_ids, src_mask)  # le dialogue n'est encodé qu'une fois

    generated = torch.tensor([[tokenizer.bos_token_id]], device=device)  # on commence par <s>
    for _ in range(max_new_tokens - 1):
        tgt_mask = torch.ones_like(generated)  # pas de padding dans ce qu'on génère
        hidden = model.decode(generated, encoder_hidden_state, src_mask, tgt_mask)
        next_token = model.summarizer_head(hidden[:, -1, :]).argmax(dim=-1, keepdim=True)  # mot suivant le plus probable
        generated = torch.cat([generated, next_token], dim=1)
        if next_token.item() == tokenizer.eos_token_id:  # le modèle a décidé de s'arrêter
            break
    return tokenizer.decode(generated[0], skip_special_tokens=True)


# Dans DialogSum, chaque dialogue du test apparaît 3 fois (avec 3 résumés de référence écrits par 3 personnes) :
# on prend les exemples 0, 3 et 6 pour avoir 3 dialogues différents.
for i in [0, 3, 6]:
    example = tiny_dataset["test"][i]
    print(f"===== Exemple de test {i} =====")
    print("Dialogue (début) :", example["dialogue"][:300].replace("\n", " "), "...")
    print("Résumé de référence :", example["summary"])
    print("Résumé du modèle    :", generate_summary(summarizer, example["dialogue"]))
    print()

💡 **Ce qu'on observe** (exemple de résultats de notre exécution ; chez toi les phrases seront différentes, mais du même genre) :

| Dialogue | Résumé de référence | Résumé généré par le modèle |
|---|---|---|
| un chef dicte une note interne interdisant les messageries instantanées | *Ms. Dawson helps #Person1# to write a memo…* | *#Person1# and #Person2# are talking about the best, and the number and the Air China's counter…* |
| quelqu'un arrive en retard à cause des embouteillages | *#Person2# arrives late because of traffic jam…* | *#Person1# wants to buy a gift for #Person2#'s younger brother…* |
| deux amis parlent du divorce d'un couple | *#Person1# tells Kate that Masha and Hero get divorced…* | *#Person1# helps #Person2# prepare for the picture…* |

**Ce que le modèle a appris :** la **forme** d'un résumé DialogSum. Il commence par `#Person1#`, écrit à la troisième personne, utilise les tournures typiques (*"are talking about"*, *"wants to"*, *"helps … to"*), et produit des phrases presque grammaticales qui s'arrêtent d'elles-mêmes (il a appris à générer `</s>`).

**Ce qu'il n'a pas appris :** le **contenu**. Les résumés n'ont aucun rapport avec les dialogues : *Air China*, *a gift for a younger brother*, *L / C* sont des morceaux des **résumés d'entraînement** recrachés presque au hasard. C'est le sur-apprentissage vu plus haut. En pratique, le décodeur se comporte comme un **modèle de langage** qui imite les résumés qu'il a vus et qui exploite à peine la cross-attention vers le dialogue.

**Pourquoi ?** Résumer demande de **comprendre** un dialogue de ~130 mots et de **sélectionner** l'essentiel, avec un seul bloc encodeur et un seul bloc décodeur, partis de zéro, et seulement 500 exemples. C'est beaucoup trop peu. À titre de comparaison, **BART** a été pré-entraîné sur 160 Go de texte avant d'être fine-tuné sur le résumé. C'est la différence entre le Notebook 2 (fine-tuning d'un modèle pré-entraîné) et ce notebook (entraînement de zéro) : le but ici est de **comprendre le mécanisme** encodeur-décodeur, masques et génération mot par mot, pas d'obtenir un bon résumeur.